# 📖 سيرفر تلاوة القرآن الكريم بالذكاء الاصطناعي — نسخة Real-time
### Quran Recitation Server — Whisper + LoRA on Google Colab GPU

**الموديل / Model:** `KheemP/whisper-base-quran-lora`

الـ Notebook ده بيشغّل موديل التلاوة على **Nvidia T4 GPU** مجاناً عبر Google Colab، ويعمل `FastAPI` server متوصل بتطبيق المصحف عن طريق رابط عام.

### ⚡ إيه الجديد في v2.0 (للظهور اللحظي للآيات)
- **WebSocket endpoint `/ws/recite`**: الموقع بيبعت الصوت أول بأول (PCM)، والسيرفر بيرد بـ **partial** (نص متغير كل ~0.7 ثانية) و**final** (نص ثابت عند كل وقفة).
- **دمج الـ LoRA في الموديل** (`merge_and_unload`) + `float16` + `SDPA` + `greedy decoding` ← أسرع بكتير من الـ pipeline القديم.
- **Warm-up** للموديل قبل أول طلب، فمفيش تأخير في أول تلاوة.
- **Endpoint الـ HTTP القديم لسه شغال** (نفس الشكل) + دعم `session_id` للدفعات (chunks) المتتالية.
- **Cloudflare Tunnel** كبديل أسرع وأثبت من LocalTunnel (ومن غير Password).

---

## 🗂️ فهرس الأقسام / Table of Contents

| # | القسم | Section | الوصف |
|---|-------|---------|-------|
| 0 | تفعيل الـ GPU | GPU Setup | خطوات يدوية قبل التشغيل |
| 1 | التحقق من الـ GPU | GPU Check | التأكد إن CUDA شغال |
| 2 | تثبيت المكتبات | Installation | ffmpeg + Python packages |
| 3 | تحميل الموديل | Model Loading | Whisper + LoRA (مدموج) + Warm-up |
| 4 | بناء الـ API | FastAPI Server | HTTP + WebSocket + إعدادات الـ real-time |
| 5 | تشغيل السيرفر | Run Server | تشغيل uvicorn في الخلفية |
| 6 | الرابط العام | Public Tunnel | Cloudflare (موصى به) / LocalTunnel / ngrok |
| 7 | اختبار | Testing | اختبار HTTP + اختبار WebSocket (محاكاة بث مباشر) |
| 8 | كود الواجهة | Frontend | مثال JavaScript للبث اللحظي من المتصفح |

> 💡 شغّل الخلايا **بالترتيب** من فوق لتحت.

---
## ⚡ 0. تفعيل الـ GPU (خطوة يدوية)

1. من القائمة العلوية اختر **Runtime** (وقت التشغيل).
2. اضغط **Change runtime type** (تغيير نوع وقت التشغيل).
3. في Hardware accelerator اختر **T4 GPU**.
4. اضغط **Save** (حفظ).

---
## 🔍 1. التحقق من الـ GPU | GPU Check

In [ ]:
import torch

cuda_ok = torch.cuda.is_available()
print("-" * 50)
print("CUDA Available:", cuda_ok)
if cuda_ok:
    print("🚀 GPU Device:", torch.cuda.get_device_name(0))
    print("🎉 رائع! السيرفر سيعمل على كارت الشاشة فائق السرعة.")
else:
    print("⚠️ تحذير: الـ GPU غير مفعل! سيعمل على الـ CPU ببطء.")
    print("يرجى الذهاب إلى: Runtime > Change runtime type > اختيار T4 GPU")
print("-" * 50)

--------------------------------------------------
CUDA Available: True
🚀 GPU Device: Tesla T4
🎉 رائع! السيرفر سيعمل على كارت الشاشة فائق السرعة.
--------------------------------------------------


---
## 📦 2. تثبيت المكتبات | Installation

- `ffmpeg`: لتحويل الصوت إلى 16kHz mono WAV.
- باقي الحزم: `fastapi`, `uvicorn`, `transformers`, `peft`, `soundfile`, `librosa`.

In [ ]:
!apt-get install -y ffmpeg
!pip install -q fastapi "uvicorn[standard]" websockets pydantic python-multipart torch transformers peft accelerate soundfile librosa nest-asyncio requests

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.


In [ ]:
!pip uninstall -y -q torchao

---
## 🧠 3. تحميل الموديل | Model Loading

بنحمّل الـ base model (Whisper) ثم نركّب عليه الـ LoRA adapter و**ندمجه** في الأوزان (`merge_and_unload`) علشان نشيل تكلفة الـ adapter وقت الاستدلال.

التحسينات المطبقة للسرعة:

| الإعداد | الفايدة |
|---------|---------|
| `float16` على الـ GPU | أسرع ~2x وذاكرة أقل |
| `merge_and_unload()` | مفيش حسابات LoRA إضافية في كل خطوة |
| `attn_implementation="sdpa"` | attention أسرع (مع fallback تلقائي) |
| `num_beams=1` (greedy) | أسرع بكتير من beam search، ومناسب للـ live |
| `torch.inference_mode()` | بدون تتبع gradients |
| `max_new_tokens` متناسب مع طول الصوت | يمنع التكرار/الهلوسة الطويلة |
| Warm-up | أول طلب حقيقي ما يتأخرش |
| بدون `pipeline(chunk_length_s=30)` | الـ chunking بيضيف overhead مالوش لازمة على مقاطع قصيرة |

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import re
import time
import threading
import numpy as np
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
from peft import PeftModel, PeftConfig

# ---------- الإعدادات / Config ----------
MODEL_ID = "KheemP/whisper-base-quran-lora"
FALLBACK_BASE = "tarteel-ai/whisper-base-ar-quran"
SAMPLE_RATE = 16000

use_cuda = torch.cuda.is_available()
device = "cuda:0" if use_cuda else "cpu"
torch_dtype = torch.float16 if use_cuda else torch.float32

# ---------- التحميل / Loading ----------
print(f"🚀 جاري تحميل موديل القرآن الكريم ({MODEL_ID}) على: {device}...")

peft_config = PeftConfig.from_pretrained(MODEL_ID)
base_model_name = peft_config.base_model_name_or_path or FALLBACK_BASE

processor = AutoProcessor.from_pretrained(base_model_name)

def _load_base(**extra):
    return AutoModelForSpeechSeq2Seq.from_pretrained(
        base_model_name, torch_dtype=torch_dtype, low_cpu_mem_usage=True, **extra
    )

try:
    base_model = _load_base(attn_implementation="sdpa")
except Exception:
    base_model = _load_base()   # نسخة transformers قديمة

model = PeftModel.from_pretrained(base_model, MODEL_ID)
model = model.merge_and_unload()                       # دمج الـ LoRA في الأوزان
model = model.to(device=device, dtype=torch_dtype).eval()

# ---------- دالة التعرف على الصوت / ASR ----------
GEN_KW = dict(language="ar", task="transcribe")
_gpu_lock = threading.Lock()   # طلب واحد على الـ GPU في نفس الوقت

def transcribe(audio: np.ndarray) -> str:
    """audio: float32 mono 16kHz  ->  نص عربي (Whisper بيقبل لحد 30 ثانية)"""
    audio = audio[-30 * SAMPLE_RATE:]
    if audio.size < int(0.2 * SAMPLE_RATE):
        return ""
    feats = processor.feature_extractor(
        audio, sampling_rate=SAMPLE_RATE, return_tensors="pt"
    ).input_features.to(device=device, dtype=torch_dtype)

    max_new = int(min(220, 20 + (audio.size / SAMPLE_RATE) * 12))
    with _gpu_lock, torch.inference_mode():
        ids = model.generate(
            input_features=feats,
            num_beams=1,
            do_sample=False,
            max_new_tokens=max_new,
            **GEN_KW,
        )
    text = processor.tokenizer.batch_decode(ids, skip_special_tokens=True)[0]
    return re.sub(r"<\|.*?\|>", "", text).strip()

# ---------- Warm-up ----------
def _warmup():
    global GEN_KW
    dummy = (np.random.randn(SAMPLE_RATE * 3) * 0.01).astype(np.float32)
    try:
        transcribe(dummy)
    except Exception as e:
        print(f"⚠️ generate بـ language/task فشل ({type(e).__name__}: {e}) — هنكمل بدونهم.")
        GEN_KW = {}
        transcribe(dummy)
    t0 = time.time()
    transcribe(dummy)
    return int((time.time() - t0) * 1000)

ms = _warmup()
print(f"✅ تم تجهيز الموديل ({MODEL_ID}) على ({device}) — زمن الاستدلال بعد الـ warm-up: ~{ms}ms")

🚀 جاري تحميل موديل القرآن الكريم (KheemP/whisper-base-quran-lora) على: cuda:0...


adapter_config.json:   0%|          | 0.00/939 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/827 [00:00<?, ?B/s]

...

adapter_model.safetensors: downloading bytes:           |  0.00B            

[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


⚠️ generate بـ language/task فشل (ValueError: The generation config is outdated and is thus not compatible with the `language` argument to `generate`. Please update the generation config as per the instructions https://github.com/huggingface/transformers/issues/25084#issuecomment-1664398224) — هنكمل بدونهم.


[transformers] Both `max_new_tokens` (=56) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=56) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ تم تجهيز الموديل (KheemP/whisper-base-quran-lora) على (cuda:0) — زمن الاستدلال بعد الـ warm-up: ~291ms


---
## 🌐 4. بناء الـ API | FastAPI Server

### الـ Endpoints

| Method | Path | الوظيفة |
|--------|------|---------|
| `GET` | `/` , `/health` | فحص حالة السيرفر وتفاصيل كارت الشاشة |
| `WS` | `/ws/recite` | **بث مباشر (Real-time)**: الواجهة تبعت PCM 16kHz، والسيرفر يرد بـ `partial` و `final` |
| `POST` | `/api/recite/analyze` | تسجيل كامل أو دفعات (chunks) — متوافق مع الواجهة القديمة |

### بروتوكول الـ WebSocket
- **من الواجهة:** frames بايناري = `int16` little-endian، mono، 16000Hz (يفضل كل ~200ms). ورسائل نصية JSON: `{"type":"flush"}` (أنهِ المقطع الحالي) و `{"type":"reset"}` (امسح).
- **من السيرفر:** `{"type":"partial","segment":N,"text":"..."}` (استبدل بيه النص الحالي) و `{"type":"final","segment":N,"text":"..."}` (ثبّته).
- **بارامترات اختيارية في الرابط:** `?vad=0.01&step=0.7&silence=0.8&max_seg=25`

### ضبط الـ Real-time (في أول الخلية)
| الإعداد | الافتراضي | معناه |
|---------|-----------|-------|
| `STEP_S` | 0.7 | كل كام ثانية صوت جديد نحدّث الـ partial (أقل = أسرع ظهور لكن حمل أعلى على الـ GPU) |
| `END_SILENCE_S` | 0.8 | سكوت بالثواني يعتبر نهاية مقطع/وقفة فنثبّت الـ final |
| `MAX_SEG_S` | 25 | أقصى طول مقطع قبل التثبيت الإجباري (Whisper حده 30 ثانية) |
| `VAD_RMS` | 0.010 | حد الطاقة لاعتبار الصوت كلام. لو الميكروفون هادي نزّله، ولو فيه دوشة ارفعه |

In [ ]:
import os
import io
import json
import time
import asyncio
import tempfile
import subprocess
import traceback
import warnings
import numpy as np
warnings.filterwarnings("ignore")

from fastapi import FastAPI, UploadFile, File, Form, HTTPException, WebSocket
from fastapi.middleware.cors import CORSMiddleware

# ---------- إعدادات الـ Real-time / Tuning ----------
STEP_S        = 0.7     # كل كام ثانية صوت جديد نعمل partial
END_SILENCE_S = 0.8     # سكوت يعتبر نهاية مقطع
MAX_SEG_S     = 25.0    # أقصى طول مقطع قبل التثبيت الإجباري
PREROLL_S     = 0.5     # صوت قبل بداية الكلام نحتفظ بيه (علشان ما نقصش أول حرف)
MIN_SPEECH_S  = 0.35    # أقل مدة كلام فعلي (يمنع هلوسة Whisper على الضوضاء)
VAD_RMS       = 0.010   # حد طاقة الصوت (0..1)

SESSION_TTL_S     = 300
MAX_SESSION_BYTES = 25 * 1024 * 1024

# ---------- إنشاء التطبيق / App ----------
app = FastAPI(
    title="Quran Recitation Whisper AI API",
    description="سيرفر التعرف على تلاوة القرآن الكريم (HTTP + WebSocket real-time)",
    version="2.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ---------- دوال مساعدة / Helpers ----------
def decode_audio_bytes(data: bytes) -> np.ndarray:
    """أي صيغة صوت (webm/ogg/mp4/wav/mp3) -> float32 mono 16kHz، من غير ملفات مؤقتة (مع fallback)"""
    cmd = ["ffmpeg", "-v", "error", "-i", "pipe:0",
           "-f", "s16le", "-acodec", "pcm_s16le", "-ar", str(SAMPLE_RATE), "-ac", "1", "pipe:1"]
    res = subprocess.run(cmd, input=data, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    out = res.stdout
    if not out:
        # بعض الصيغ (mp4/m4a) محتاجة ملف قابل للـ seek
        with tempfile.NamedTemporaryFile(delete=False, suffix=".bin") as f:
            f.write(data)
            path = f.name
        try:
            cmd[cmd.index("pipe:0")] = path
            res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
            out = res.stdout
        finally:
            try:
                os.remove(path)
            except Exception:
                pass
    if not out:
        raise ValueError("تعذّر فك تشفير الصوت: " + res.stderr.decode(errors="ignore")[-200:])
    return np.frombuffer(out[: len(out) // 2 * 2], dtype=np.int16).astype(np.float32) / 32768.0

def rms(x: np.ndarray) -> float:
    return float(np.sqrt(np.mean(x * x))) if x.size else 0.0

def transcribe_long(pcm: np.ndarray) -> str:
    """للملفات الأطول من 30 ثانية: نقسمها لنوافذ 28 ثانية"""
    win = 28 * SAMPLE_RATE
    parts = (transcribe(pcm[i:i + win]) for i in range(0, len(pcm), win))
    return " ".join(p for p in parts if p)

SESSIONS = {}   # session_id -> {"bytes": bytearray, "ts": float}

def _gc_sessions():
    now = time.time()
    for sid in [k for k, v in SESSIONS.items() if now - v["ts"] > SESSION_TTL_S]:
        SESSIONS.pop(sid, None)

# ---------- Endpoints ----------
@app.get("/")
@app.get("/health")
def health():
    return {
        "status": "ok",
        "model": MODEL_ID,
        "device": device,
        "dtype": str(torch_dtype),
        "websocket": "/ws/recite",
        "message": "خادم تلاوة القرآن جاهز للعمل على Google Colab GPU"
    }

@app.post("/api/recite/analyze")
async def analyze_recitation(
    audio: UploadFile = File(...),
    page_number: int = Form(1),
    reference_text: str = Form(""),
    session_id: str = Form(""),
    is_final: bool = Form(False),
):
    """
    - بدون session_id: ملف تسجيل كامل -> نص.
    - مع session_id: دفعات MediaRecorder المتتالية (الدفعة الأولى فيها الـ header، فلازم نجمعهم).
      كل طلب بيرجّع النص لآخر ~25 ثانية. ابعت is_final=true في آخر دفعة لتنظيف الجلسة.
    """
    t0 = time.time()
    try:
        data = await audio.read()

        if session_id:
            _gc_sessions()
            s = SESSIONS.setdefault(session_id, {"bytes": bytearray(), "ts": time.time()})
            s["bytes"].extend(data)
            s["ts"] = time.time()
            if len(s["bytes"]) > MAX_SESSION_BYTES:
                SESSIONS.pop(session_id, None)
                raise HTTPException(status_code=413, detail="الجلسة طويلة جداً، ابدأ session_id جديد")
            payload = bytes(s["bytes"])
            if is_final:
                SESSIONS.pop(session_id, None)
        else:
            payload = data

        if len(payload) < 100:
            return {"status": "success", "transcription": "", "page_number": page_number}

        pcm = await asyncio.to_thread(decode_audio_bytes, payload)

        if rms(pcm) < 0.002:     # صمت تام -> ما نشغّلش الموديل (بيهلوس)
            text = ""
        elif session_id:
            text = await asyncio.to_thread(transcribe, pcm[-25 * SAMPLE_RATE:])
        else:
            text = await asyncio.to_thread(transcribe_long, pcm)

        latency = int((time.time() - t0) * 1000)
        print(f"✨ [{latency}ms] {text}")
        return {
            "status": "success",
            "transcription": text,
            "page_number": page_number,
            "model": MODEL_ID,
            "latency_ms": latency,
            "partial": bool(session_id and not is_final),
        }

    except HTTPException:
        raise
    except Exception as e:
        print("❌ خطأ أثناء معالجة التلاوة:")
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))

@app.websocket("/ws/recite")
async def ws_recite(ws: WebSocket):
    await ws.accept()

    qp = ws.query_params
    def _q(key, default):
        try:
            return float(qp.get(key, default))
        except (TypeError, ValueError):
            return default
    vad      = _q("vad", VAD_RMS)
    step_s   = _q("step", STEP_S)
    end_sil  = _q("silence", END_SILENCE_S)
    max_seg  = min(_q("max_seg", MAX_SEG_S), 28.0)

    buf = np.zeros(0, dtype=np.float32)   # المقطع الحالي
    heard = False
    silence_s = 0.0
    voiced_s = 0.0
    last_emit = 0                          # طول الـ buffer وقت آخر partial
    segment = 1

    async def run_asr(audio):
        t0 = time.time()
        text = await asyncio.to_thread(transcribe, audio)
        return text, int((time.time() - t0) * 1000)

    def reset_state():
        nonlocal buf, heard, silence_s, voiced_s, last_emit
        buf = np.zeros(0, dtype=np.float32)
        heard, silence_s, voiced_s, last_emit = False, 0.0, 0.0, 0

    async def finalize():
        nonlocal segment
        if heard and voiced_s >= MIN_SPEECH_S and buf.size:
            text, ms = await run_asr(buf)
            if text:
                await ws.send_json({"type": "final", "segment": segment, "text": text, "latency_ms": ms})
                segment += 1
        reset_state()

    try:
        await ws.send_json({"type": "ready", "sample_rate": SAMPLE_RATE, "format": "pcm_s16le_mono"})
        while True:
            msg = await ws.receive()
            if msg["type"] == "websocket.disconnect":
                break

            # ---- رسائل التحكم ----
            if msg.get("text") is not None:
                try:
                    cmd = json.loads(msg["text"]).get("type")
                except Exception:
                    cmd = None
                if cmd == "flush":
                    await finalize()
                elif cmd == "reset":
                    reset_state()
                continue

            data = msg.get("bytes")
            if not data:
                continue

            # ---- صوت ----
            chunk = np.frombuffer(data[: len(data) // 2 * 2], dtype=np.int16).astype(np.float32) / 32768.0
            dur = chunk.size / SAMPLE_RATE
            buf = np.concatenate([buf, chunk])

            if rms(chunk) > vad:
                heard = True
                silence_s = 0.0
                voiced_s += dur
            elif heard:
                silence_s += dur

            if not heard:
                keep = int(PREROLL_S * SAMPLE_RATE)
                if buf.size > keep:
                    buf = buf[-keep:]
                continue

            if silence_s >= end_sil or buf.size / SAMPLE_RATE >= max_seg:
                await finalize()
            elif (buf.size - last_emit) / SAMPLE_RATE >= step_s and voiced_s >= MIN_SPEECH_S:
                last_emit = buf.size
                text, ms = await run_asr(buf)
                if text:
                    await ws.send_json({"type": "partial", "segment": segment, "text": text, "latency_ms": ms})

    except Exception as e:
        # غالباً الواجهة قفلت الاتصال
        if "disconnect" not in repr(e).lower() and "close" not in repr(e).lower():
            traceback.print_exc()

print("✅ تم بناء الـ API بنجاح (HTTP + WebSocket).")

✅ تم بناء الـ API بنجاح (HTTP + WebSocket).


---
## ▶️ 5. تشغيل السيرفر | Run Server

بنشغّل `uvicorn` في **Thread** منفصل (خلفية) علشان الخلايا اللي بعد كده تفضل شغالة وما تعلقش، مع تفعيل دعم الـ WebSocket وضبط الـ keep-alive للاتصالات الطويلة.

In [ ]:
import time
import threading
import urllib.request
import nest_asyncio
import uvicorn

PORT = 8000
nest_asyncio.apply()

config = uvicorn.Config(
    app, host="0.0.0.0", port=PORT, log_level="warning",
    ws="websockets", ws_ping_interval=20, ws_ping_timeout=30,
    timeout_keep_alive=60,
)
server = uvicorn.Server(config)
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()

# استنى لحد ما السيرفر يرد (بدل sleep ثابت)
for _ in range(30):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=1)
        break
    except Exception:
        time.sleep(0.5)

print(f"✅ السيرفر شغال على المنفذ {PORT} في الخلفية (HTTP + WebSocket).")

✅ السيرفر شغال على المنفذ 8000 في الخلفية (HTTP + WebSocket).


---
## 🔗 6. الرابط العام | Public Tunnel

> 💡 **للـ real-time الأفضل تستخدم Cloudflare Tunnel (القسم 6.3)**: بيدعم الـ WebSocket بثبات، وما بيطلبش Password، وغالباً latency أقل من LocalTunnel.

### 6.1 الحصول على كلمة مرور الـ Tunnel (لـ LocalTunnel بس)
عند فتح رابط `loca.lt` أول مرة بيطلب **Tunnel Password**، وهي الـ IP الخارجي لـ Colab. الخلية دي بتطبعها لتنسخها بسهولة.

In [ ]:
import urllib.request

try:
    colab_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
    print("=" * 50)
    print(f"🔑 كلمة مرور الرابط (Tunnel Password / Endpoint IP): {colab_ip}")
    print("=" * 50)
except Exception as e:
    colab_ip = None
    print("⚠️ تعذّر جلب الـ IP:", e)

🔑 كلمة مرور الرابط (Tunnel Password / Endpoint IP): 34.91.239.6


### 6.2 تشغيل LocalTunnel (بديل)

بعد تشغيل الخلية، انسخ الرابط اللي بينتهي بـ `.loca.lt` وحطه في إعدادات التلاوة في الموقع.

> ⚠️ الخلية دي **بتفضل شغالة** (blocking) طول ما الرابط مطلوب. متوقفهاش.
> ⚠️ صفحة الـ Password بتاعة LocalTunnel ممكن تعطّل اتصال الـ WebSocket من المتصفح (مفيش طريقة تبعت header مخصوص مع `new WebSocket`). لو حصل كده استخدم 6.3.

In [ ]:
print("🚀 جاري تشغيل LocalTunnel لإنشاء رابط HTTPS عام...\n")
!npx -y localtunnel --port 8000

🚀 جاري تشغيل LocalTunnel لإنشاء رابط HTTPS عام...

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇your url is: https://breezy-cooks-give.loca.lt


[transformers] Both `max_new_tokens` (=31) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✨ [1837ms] فَخَرَ


[transformers] Both `max_new_tokens` (=67) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✨ [1066ms] إِنَّ الَّذِينَ جَادَ


...

✨ [6060ms] إِنَّ الَّذِينَ يُحِبُّونَ أَن تَشِيعَ الْفَاحِشَةُ فِي الَّذِينَ آمَنُوا لَهُمْ عَذَابٌ أَلِيمٌ فِي الدُّنْيَا وَالْآخِرَةِ ۗ وَاللَّهُ يَعْلَمُ وَأَنتُمْ لَا تَعْلَمُونَ ۗ وَلَوْلَا فَضْلُ اللَّهِ عَلَىٰ أَلَيْكُمْ وَرَحْمَتُهُ وَأَنَّ اللَّهَ رَءُوفٌ رَّحِيمٌ يَا أَيُّهَا الَّذِينَ آمَنُوا لَا تَتَّبِعُوا خُطُوَاتِ الشَّيْطَانِ ۖ وَمَن يَتَّبِعْ خُطُوَاتِ الشَّيْطَانِ فَإِنَّهُ يَا يَأْمُرُ بِالْفَحْشَاءِ وَالْمُنكَرِ ۚ وَلَوْلَا فَضْلُ اللَّهِ عَلَيْكُمْ وَرَحْمَتُهُ مَا زَكَىٰ مِنكُم مِّنْ أَحَدٍ أَبَدًا ۚ وَلَٰكِنَّ اللَّهَ يُزَكِّي مَن يَشَاءُ ۗ وَاللَّهُ سَمِيعٌ عَلِيمٌ وَلَا يَأْتَلِ أُولُو الْفَضْلِ مِنكُمْ وَالسَّعَةِ أَن يُؤْتُوا أُولِي الْقُرْبَىٰ وَالْمَسَاكِينَ وَالْمُهَاجِرِينَ فِي سَبِيلٍ


[transformers] Both `max_new_tokens` (=220) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=220) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=220) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=220) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

✨ [6638ms] إِنَّ الَّذِينَ يُحِبُّونَ أَن تَشِيعَ الْفَاحِشَةُ فِي الَّذِينَ آمَنُوا لَهُمْ عَذَابٌ أَلِيمٌ فِي الدُّنْيَا وَالْآخِرَةِ ۗ وَاللَّهُ يَعْلَمُ وَأَنتُمْ لَا تَعْلَمُونَ ۗ وَلَوْلَا فَضْلُ اللَّهِ عَلَىٰ أَلَيْكُمْ وَرَحْمَتُهُ وَأَنَّ اللَّهَ رَءُوفٌ رَّحِيمٌ يَا أَيُّهَا الَّذِينَ آمَنُوا لَا تَتَّبِعُوا خُطُوَاتِ الشَّيْطَانِ ۖ وَمَن يَتَّبِعْ خُطُوَاتِ الشَّيْطَانِ فَإِنَّهُ يَا يَأْمُرُ بِالْفَحْشَاءِ وَالْمُنكَرِ ۚ وَلَوْلَا فَضْلُ اللَّهِ عَلَيْكُمْ وَرَحْمَتُهُ مَا زَكَىٰ مِنكُم مِّنْ أَحَدٍ أَبَدًا ۚ وَلَٰكِنَّ اللَّهَ يُزَكِّي مَن يَشَاءُ ۗ وَاللَّهُ سَمِيعٌ عَلِيمٌ وَلَا يَأْتَلِ أُولُو الْفَضْلِ مِنكُمْ وَالسَّعَةِ أَن يُؤْتُوا أُولِي الْقُرْبَىٰ وَالْمَسَاكِينَ وَالْمُهَاجِرِينَ فِي سَبِيلِ اللَّهِ وَلْيَعْفُوا وَلْيَصْفَحُوا أَلَا تُحِبُّونَ أَن يَشْتَرِ


[transformers] Both `max_new_tokens` (=220) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=220) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=220) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=220) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

✨ [6148ms] إِنَّ الَّذِينَ يُحِبُّونَ أَن تَشِيعَ الْفَاحِشَةُ فِي الَّذِينَ آمَنُوا لَهُمْ عَذَابٌ أَلِيمٌ فِي الدُّنْيَا وَالْآخِرَةِ ۗ وَاللَّهُ يَعْلَمُ وَأَنتُمْ لَا تَعْلَمُونَ ۗ وَلَوْلَا فَضْلُ اللَّهِ عَلَىٰ أَلَيْكُمْ وَرَحْمَتُهُ وَأَنَّ اللَّهَ رَءُوفٌ رَّحِيمٌ يَا أَيُّهَا الَّذِينَ آمَنُوا لَا تَتَّبِعُوا خُطُوَاتِ الشَّيْطَانِ ۖ وَمَن يَتَّبِعْ خُطُوَاتِ الشَّيْطَانِ فَإِنَّهُ يَا يَأْمُرُ بِالْفَحْشَاءِ وَالْمُنكَرِ ۚ وَلَوْلَا فَضْلُ اللَّهِ عَلَيْكُمْ وَرَحْمَتُهُ مَا زَكَىٰ مِنكُم مِّنْ أَحَدٍ أَبَدًا ۚ وَلَٰكِنَّ اللَّهَ يُزَكِّي مَن يَشَاءُ ۗ وَاللَّهُ سَمِيعٌ عَلِيمٌ وَلَا يَأْتَلِ أُولُو الْفَضْلِ مِنكُمْ وَالسَّعَةِ أَن يُؤْتُوا أُولِي الْقُرْبَىٰ وَالْمَسَاكِينَ وَالْمُهَاجِرِينَ فِي سَبِيلِ اللَّهِ وَلْيَعْفُوا وَلْيَصْفَحُوا أَلَا تُحِبُّونَ أَن يَغْفِرَ اللَّهُ لَكُمْ ۗ وَاللَّهُ غَفُورٌ رَّحِيمٌ
^C


### 6.3 ⭐ Cloudflare Tunnel (موصى به للـ real-time)

الخلية دي **مش blocking**: بتنزّل `cloudflared` وتشغّله في الخلفية وتطبع الرابطين (HTTP و WebSocket). لو اتغيّر الرابط (بعد إعادة تشغيل الخلية) حدّثه في الموقع.

In [ ]:
import os, re, time, threading, subprocess

CF_BIN = os.path.abspath("cloudflared")
if not os.path.exists(CF_BIN):
    subprocess.run(
        ["wget", "-q", "-O", CF_BIN,
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
        check=True,
    )
    os.chmod(CF_BIN, 0o755)

# لو فيه نسخة شغالة قبل كده، نقفلها
try:
    cf_proc.terminate()
except Exception:
    pass

cf_proc = subprocess.Popen(
    [CF_BIN, "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate", "--protocol", "http2"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

_cf = {"url": None}
def _drain():
    # لازم نفضّي الـ pipe باستمرار وإلا cloudflared هيتعلق
    for line in cf_proc.stdout:
        m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
        if m and not _cf["url"]:
            _cf["url"] = m.group(0)
threading.Thread(target=_drain, daemon=True).start()

for _ in range(80):
    if _cf["url"]:
        break
    time.sleep(0.5)

if _cf["url"]:
    base = _cf["url"]
    print("=" * 60)
    print("🌐 HTTP :", base + "/api/recite/analyze")
    print("⚡ WS   :", base.replace("https://", "wss://") + "/ws/recite")
    print("=" * 60)
else:
    print("⚠️ ما قدرتش أجيب الرابط. تأكد إن الإنترنت شغال في Colab وأعد تشغيل الخلية.")

### 6.4 (اختياري) بديل: ngrok
لو عايز رابط بديل عبر ngrok، فك التعليق (uncomment) وضع الـ token بتاعك:


In [ ]:
# !pip install -q pyngrok
# from pyngrok import ngrok
#
# NGROK_TOKEN = "ضع_رمز_ngrok_هنا"
# ngrok.set_auth_token(NGROK_TOKEN)
# tunnel = ngrok.connect(8000)
# print("🔗 رابط السيرفر على ngrok:", tunnel.public_url)

---
## 🧪 7. اختبار السيرفر | Testing

اختبار محلي سريع للتأكد من استجابة السيرفر:

### 7.1 اختبار فحص الحالة (/health)

In [ ]:
import requests

r = requests.get("http://127.0.0.1:8000/health")
print("Status:", r.status_code)
print("Response:", r.json())

### 7.2 اختبار تحليل ملف صوتي
ارفع ملف تلاوة (مثلاً `test.mp3`) من لوحة الملفات على اليسار، وعدّل اسم الملف:

In [9]:
import requests

AUDIO_FILE = "elkiwitiTrimmed.mp3"   # ← غيّر اسم الملف إذا رغبت بالاختبار

try:
    with open(AUDIO_FILE, "rb") as f:
        r = requests.post(
            "http://127.0.0.1:8000/api/recite/analyze",
            files={"audio": (AUDIO_FILE, f)},
            data={"page_number": 1, "reference_text": ""}
        )
    print("Status:", r.status_code)
    print("Result:", r.json())
except FileNotFoundError:
    print(f"ℹ️ لا يوجد ملف باسم {AUDIO_FILE}، يمكنك رفع ملف لاختباره أو التلاوة مباشرة من المتصفح.")

[transformers] Both `max_new_tokens` (=220) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✨ [1191ms] ۞ وَاشْتَدَا الْغَوَّانُ وَأْوِيحَوْفٌ لِلَّهِ مُعَالَمِينَ
Status: 200
Result: {'status': 'success', 'transcription': '۞ وَاشْتَدَا الْغَوَّانُ وَأْوِيحَوْفٌ لِلَّهِ مُعَالَمِينَ', 'page_number': 1, 'model': 'KheemP/whisper-base-quran-lora', 'latency_ms': 1191, 'partial': False}


### 7.3 اختبار الـ WebSocket (محاكاة بث مباشر)

الخلية دي بتقرأ ملف الصوت وتبعته على شكل frames (200ms) بنفس سرعة الكلام الحقيقية، وتطبع الـ `partial` و `final` أول ما توصل، مع زمن الاستجابة.

In [ ]:
import asyncio, json, time
import numpy as np
import websockets

WS_AUDIO = "test.mp3"      # ← غيّر اسم الملف
SPEED = 1.0                # 1.0 = سرعة حقيقية، 2.0 = أسرع مرتين (للاختبار)

async def stream_test(path=WS_AUDIO, chunk_ms=200, speed=SPEED):
    try:
        with open(path, "rb") as f:
            pcm = decode_audio_bytes(f.read())
    except FileNotFoundError:
        print(f"ℹ️ ارفع ملف صوت باسم {path} الأول.")
        return
    pcm = np.concatenate([pcm, np.zeros(int(1.5 * SAMPLE_RATE), dtype=np.float32)])  # سكوت في الآخر
    i16 = (np.clip(pcm, -1, 1) * 32767).astype(np.int16)
    step = int(SAMPLE_RATE * chunk_ms / 1000)

    async with websockets.connect(f"ws://127.0.0.1:{PORT}/ws/recite") as ws:
        t_start = time.time()

        async def reader():
            async for raw in ws:
                m = json.loads(raw)
                if m["type"] in ("partial", "final"):
                    print(f"[{time.time()-t_start:5.1f}s] {m['type']:7s} seg{m['segment']} ({m['latency_ms']}ms): {m['text']}")

        task = asyncio.create_task(reader())
        for i in range(0, len(i16), step):
            await ws.send(i16[i:i + step].tobytes())
            await asyncio.sleep(chunk_ms / 1000 / speed)
        await ws.send(json.dumps({"type": "flush"}))
        await asyncio.sleep(1.5)
        task.cancel()

await stream_test()

---
## 🖥️ 8. كود الواجهة | Frontend (JavaScript)

مثال بيلتقط الميكروفون، يحوّله 16kHz PCM، ويبعته للسيرفر، ويستقبل `partial`/`final`. (شغّال في Next.js كـ client component؛ استبدل الرابط برابط Cloudflare.)

```js
const WS_URL = "wss://YOUR-TUNNEL.trycloudflare.com/ws/recite";   // + ?vad=0.01&step=0.7 لو عايز تضبط

export async function startLiveRecitation({ onPartial, onFinal }) {
  const ws = new WebSocket(WS_URL);
  ws.binaryType = "arraybuffer";
  ws.onmessage = (e) => {
    const m = JSON.parse(e.data);
    if (m.type === "partial") onPartial(m.segment, m.text);   // استبدل نص المقطع الحالي
    if (m.type === "final")   onFinal(m.segment, m.text);     // ثبّته
  };
  await new Promise((res, rej) => { ws.onopen = res; ws.onerror = rej; });

  const stream = await navigator.mediaDevices.getUserMedia({
    audio: { channelCount: 1, echoCancellation: true, noiseSuppression: true, autoGainControl: true },
  });
  const ctx = new AudioContext({ sampleRate: 16000 });          // المتصفح بيعمل الـ resample
  const workletCode = `class P extends AudioWorkletProcessor {
    process(inputs){ const c = inputs[0][0]; if (c) this.port.postMessage(c.slice(0)); return true; }
  } registerProcessor("pcm", P);`;
  await ctx.audioWorklet.addModule(URL.createObjectURL(new Blob([workletCode], { type: "application/javascript" })));

  const node = new AudioWorkletNode(ctx, "pcm");
  let acc = [], n = 0;
  node.port.onmessage = (e) => {
    acc.push(e.data); n += e.data.length;
    if (n < 3200) return;                                       // ~200ms
    const f = new Float32Array(n); let o = 0;
    for (const a of acc) { f.set(a, o); o += a.length; }
    acc = []; n = 0;
    const i16 = new Int16Array(f.length);
    for (let k = 0; k < f.length; k++) { const s = Math.max(-1, Math.min(1, f[k])); i16[k] = s < 0 ? s * 32768 : s * 32767; }
    if (ws.readyState === 1) ws.send(i16.buffer);
  };
  ctx.createMediaStreamSource(stream).connect(node);

  return function stop() {                                      // نادِ عليها لما المستخدم يوقف
    if (ws.readyState === 1) ws.send(JSON.stringify({ type: "flush" }));
    stream.getTracks().forEach((t) => t.stop());
    ctx.close();
    setTimeout(() => ws.close(), 1500);
  };
}
```

**ملاحظات:**
- الـ `partial` ممكن يتعدّل من رد للتاني (ده طبيعي)، فاعرضه كنص مؤقت وطابقه مع نص الصفحة، والـ `final` هو اللي تثبّته.
- استخدم الـ HTTPS/WSS دايماً (الميكروفون ما بيشتغلش على HTTP غير آمن).
- لو بتستخدم `MediaRecorder` بدل ده: ابعت كل الدفعات لـ `/api/recite/analyze` بنفس `session_id` (ومعاها `is_final=true` في آخر دفعة).

---
## 🛠️ حل المشاكل الشائعة | Troubleshooting

| المشكلة | السبب | الحل |
|---------|-------|------|
| `CUDA Available: False` | الـ GPU غير مفعل في وقت التشغيل | اذهب إلى Runtime > Change runtime type واختر T4 GPU |
| الرابط بيطلب Password | تأكيد أمان أداة localtunnel | انسخ الـ IP المطبوع في الخلية 6.1 واضغط Submit في المتصفح |
| `Address already in use` | المنفذ 8000 مستخدم مسبقاً | Runtime ← Restart session ثم أعد تشغيل الخلايا بالترتيب |
| الرابط توقف بعد فترة | Google Colab يفصل بعد فترة خمول | أعد تشغيل الخلايا 5 و 6.2 لتجديد الرابط |
| التحذيرات البرتقالية | إصدارات transformers | الكود مزود بـ `warnings.filterwarnings('ignore')` لإخفائها بالكامل |
| الـ WebSocket بيقفل أو مش بيتوصل | LocalTunnel أو رابط `http` بدل `wss` | استخدم Cloudflare Tunnel (6.3) وارجع للرابط بـ `wss://` |
| الـ partial بيتأخر | `STEP_S` كبير أو الـ GPU مش شغال | قلّل `step` (مثلاً 0.5) وتأكد من `CUDA Available: True` |
| بيقطع المقطع بدري أو متأخر | حساسية السكوت | عدّل `silence` (أقل = تثبيت أسرع) و `vad` حسب الميكروفون |
| بيطلع نص والمستخدم ساكت | `VAD_RMS` واطي بالنسبة للدوشة | ارفع `vad` (مثلاً 0.02) |
